# (Quick) Python introduction and POA calculation

<table>
  <tr>
    <td><img src="https://raw.githubusercontent.com/Mohamed-ENNHIRI/ENER958/main/notebooks/logos/solaracademy.png" width="200" />
    <td><img src="https://raw.githubusercontent.com/Mohamed-ENNHIRI/ENER958/main/notebooks/logos/usmb.png" width="200" />
    <td><img src="https://raw.githubusercontent.com/Mohamed-ENNHIRI/ENER958/main/notebooks/logos/cnrs.png" width="80" />
  </tr>
</table>

<sub>Author: Mohamed Ennhiri, PhD student, LOCIE UMR CNRS 5271, Université Savoie Mont Blanc</sub>

<sub>Structure and format inspired by the course given by Alexandre Mathieu at the Solar Academy in 2024, "PV failure detection based on operational time series". Physics and data: pvlib, PVGIS, ODRE, RTE.</sub>

This notebook introduces how to use python to explore data and calculate the Plane Of Array (POA) irradiance components.

It is broken down into two parts:

1. **Python introduction**: The student will follow the python introduction and is invited to execute the cells.
2. **POA calculation exercice**: The student is invited to calculate the POA irradiance for one timestep, then for a whole year.

The students are expected to do the **exercices**.

## I. Python introduction

Python is a programming language which enables to perform a large number of tasks, from math operations to data treatment.

This section provides useful commands when coding in Python.

### I.1 Hello world

Printing texts is necessary to show results or the progress of a script.

In [ ]:
# This is a code cell, and I am writing a comment which is not going to be executed thanks to the "#" character
# "#" is very useful to comment and precise what the code does, feel free to use it !
print("Hello world") # I print "Hello world"

In [ ]:
my_text = "Something" # Write something here. It must be surrounded with "", otherwise it is going to crash
print(my_text)

In [ ]:
# Include a variable in a pre-defined text with an f-string
site = "Le Bourget-du-Lac"
print(f"The course site is {site}")

### I.2 Operations

Basic operations are fast in Python.

In [ ]:
a = 1 # assign 1 to the variable "a"
b = 2 # assign 2 to the variable "b"
print(a + b)
print(a / b)
print(a * b)
print(a ** b) # a to the power b

In [ ]:
# Open-source libraries (with their functions) such as numpy have useful functions we can use
import numpy as np # Import numpy and rename it "np"

# Mathematical constants
print(np.pi)

# Trigonometric functions
angle = np.pi / 4
print(np.sin(angle)) # np.sin() expects radians !
print(np.cos(np.radians(45))) # np.radians() converts degrees into radians
print(np.degrees(np.arccos(0.5))) # np.degrees() converts radians into degrees

#### Exercice

Calculate the direct horizontal irradiance BHI = DNI · cos(θz) if DNI = 850 W/m² and the sun elevation is h = 67.4° (zenith θz = 90° − h). Round the result to one decimal.

In [ ]:
dni = 850 # W/m2
elevation = 67.4 # degrees
zenith = 90 - elevation # degrees
bhi = dni * np.cos(np.radians(zenith)) # W/m2, np.cos() expects radians
bhi_rounded = round(bhi, 1) # Round the decimals
print(f"BHI: {bhi_rounded} W/m²")

### I.3 List, Series and DataFrame

One way to arrange data is to use a list, a pandas Series or a DataFrame.

In [ ]:
my_list = [1, 2, 3]
print(my_list)

In [ ]:
import pandas as pd # Import the package pandas and rename it "pd"

serie = pd.Series([1, 2, 3]) # Create a Series
serie2 = pd.Series([4, 5, 6]) # Create another Series

print("My first serie")
print(serie)
print("\nMy second serie")
print(serie2)

In [ ]:
# A Series is more structured than a list and enables to make the calculations explicitly easier
print(serie + serie2)
print(serie / serie2)

In [ ]:
# You can still apply np operations on a pd.Series !
np.cos(serie)

In [ ]:
# A Series is defined by its index and its value can be extracted with the loc command
print(serie.loc[0])

In [ ]:
# Pandas Series and DataFrame are particularly used with time series, with the index being dates or times
index = pd.date_range("2023-06-21", freq="h", periods=3) # "h" for hour, "D" can also be used for day
serie_ts = pd.Series([1, 2, 3], index=index)
print(serie_ts)

In [ ]:
# And we can plot it (which is quite useful to explore data)
serie_ts.plot(marker="o")

In [ ]:
# We can also use a DataFrame to store several Series in different columns (if they have the same index)
data_df = pd.DataFrame() # Create an empty DataFrame to store the Series
data_df["first_column"] = serie
data_df["second_column"] = serie2
print(data_df)

# The same operations like np.cos(), np.sin() can be done on a column
print(np.sin(data_df["second_column"]))

# And plot it !
data_df.plot()

#### Exercice

Create a DataFrame:

- with a DAILY datetime index starting from 2023-06-19 until 2023-06-25, included,
- two columns, the second one equal to the double of the first one,

and plot it.

In [ ]:
# Create the daily index with the "freq" argument
index = pd.date_range("2023-06-19", "2023-06-25", freq="D")

# Create a first time series with some values
serie_ts = pd.Series([1, 2, 3, 4, 5, 6, 7], index=index)

# Create the DataFrame with the two columns
df = pd.DataFrame()
df["first_column"] = serie_ts
df["second_column"] = serie_ts * 2 # The double of the first one

df.plot(marker="o")

## II. Exercices: POA calculation

Let’s use Python to compute the POA components in a fast and (always) accurate way.

Reminder, the global POA irradiance is broken down into 3 components:

$POA = POA_{direct} + POA_{diffuse} + POA_{reflected}$

- $POA_{direct} = DNI \cdot \cos(AOI)$
- $POA_{diffuse} = DHI \cdot \frac{1 + \cos(\beta)}{2}$ under the isotropic assumption
- $POA_{reflected} = GHI \cdot \rho \cdot \frac{1 - \cos(\beta)}{2}$

with $\cos(AOI) = \cos(\theta_z)\cos(\beta) + \sin(\theta_z)\sin(\beta)\cos(\gamma_s - \gamma)$, $\beta$ the tilt, $\gamma$ the installation azimuth, $\theta_z$ the solar zenith and $\gamma_s$ the solar azimuth (clockwise from north).

In [ ]:
# Please execute this cell to import packages
import numpy as np # For math operations
import pandas as pd # To create indexed time series
import matplotlib.pyplot as plt # To plot figures

# In Google Colab, pvlib is not installed by default: the line below installs it (takes 30 s)
!pip install -q pvlib
import pvlib
print("pvlib", pvlib.__version__)

### II.1 Calculate POA for one timestep

- Under the isotropic / no-shading assumption
- With the values of the worked hour of the course (21 June, 12:00 UTC, course site)

In [ ]:
# Installation orientation
beta = 30 # tilt [°]
azimuth = 180 # azimuth [°], south

# Albedo
rho = 0.2

# Irradiance values
DNI = 850 # W/m2
GHI = 904.5 # W/m2
DHI = 120 # W/m2

# Solar position
s_zenith = 22.64 # zenith [°]
s_azimuth = 193.3 # azimuth [°]

#### Exercice

Calculate the three components and the global POA irradiance, the explicit way, with the formulas above. Expected result: 962.7 W/m².

In [ ]:
# The explicit way !

# Calculate the cos(AOI) first, np.cos() and np.sin() expect radians
cos_aoi = np.cos(np.radians(s_zenith)) * np.cos(np.radians(beta)) + np.sin(np.radians(s_zenith)) * np.sin(np.radians(beta)) * np.cos(np.radians(s_azimuth - azimuth))

POA_direct = DNI * cos_aoi # Direct component
POA_diffuse = DHI * (1 + np.cos(np.radians(beta))) / 2 # Diffuse component, isotropic sky
POA_reflected = GHI * rho * (1 - np.cos(np.radians(beta))) / 2 # Reflected component

POA = POA_direct + POA_diffuse + POA_reflected
print(f"Direct: {round(POA_direct, 1)} W/m², diffuse: {round(POA_diffuse, 1)} W/m², reflected: {round(POA_reflected, 1)} W/m²")
print(f"POA: {round(POA, 1)} W/m²")

In [ ]:
# The lazy way with the function "get_total_irradiance" from the pvlib package
from pvlib.irradiance import get_total_irradiance

# Notice it takes degrees as inputs, and the isotropic model is the default one
results = get_total_irradiance(beta, azimuth, s_zenith, s_azimuth, DNI, GHI, DHI, albedo=rho)
print(results) # A dictionary with the components

### II.2 Calculate POA for one year from satellite data

In [ ]:
# Load the satellite weather data of the course site from an online file: GHI, DNI, DHI, air temperature, wind speed
# Site: Savoie, near Le Bourget-du-Lac (Lat 45.64 N, Long 5.92 E). Source: PVGIS SARAH3, year 2023, hourly.
urlw = "https://raw.githubusercontent.com/Mohamed-ENNHIRI/ENER958/main/data/weather_45.64N_5.92E_2023.csv"
weather_data = pd.read_csv(urlw, index_col=0) # Import the csv file and use the first column as index
weather_data.index = pd.to_datetime(weather_data.index) # Convert the index into datetimes (UTC)
print(weather_data.head(5)) # Show the first five lines

In [ ]:
# Quick visualization: one clear week of June
fig, ax = plt.subplots(figsize=(9, 4)) # Create a new figure with a predefined size
weather_data.loc["2023-06-19":"2023-06-25", ["ghi", "dni", "dhi"]].plot(ax=ax, ylabel="Irradiance [W/m²]", title="Irradiance components as function of time")

In [ ]:
# Load the solar position (zenith, azimuth, elevation) calculated with the NREL algorithm:
# I. Reda and A. Andreas, Solar position algorithm for solar radiation applications. Solar Energy, 76(5), 2004.
urls = "https://raw.githubusercontent.com/Mohamed-ENNHIRI/ENER958/main/data/solarpos_45.64N_5.92E_2023.csv"
solar_position = pd.read_csv(urls, index_col=0)
solar_position.index = pd.to_datetime(solar_position.index)
print(solar_position.head(5))

#### Exercice

Calculate the POA global irradiance for the whole year, the explicit way (same formulas, applied on the whole columns), then the lazy way with pvlib. Plot both over the week of 19 to 25 June and print the annual POA energy in kWh/m² (sum of the hourly values divided by 1000).

Advice: clip cos(AOI) at 0 with `.clip(lower=0)` before multiplying by the DNI, otherwise the direct component is negative when the sun is behind the plane.

In [ ]:
# The explicit way
zenith = solar_position["zenith"] # pd.Series over the year
s_azimuth = solar_position["azimuth"]

# Calculate the cos(AOI) first, then remove the negative values (sun behind the plane)
cos_aoi = np.cos(np.radians(zenith)) * np.cos(np.radians(beta)) + np.sin(np.radians(zenith)) * np.sin(np.radians(beta)) * np.cos(np.radians(s_azimuth - azimuth))
cos_aoi = cos_aoi.clip(lower=0)

# Calculate all components and sum them up into "POA_explicit"
POA_direct = weather_data["dni"] * cos_aoi
POA_diffuse = weather_data["dhi"] * (1 + np.cos(np.radians(beta))) / 2
POA_reflected = weather_data["ghi"] * rho * (1 - np.cos(np.radians(beta))) / 2
POA_explicit = POA_direct + POA_diffuse + POA_reflected

# The lazy way with the pvlib package
data = get_total_irradiance(beta, azimuth, solar_position["zenith"], solar_position["azimuth"], weather_data["dni"], weather_data["ghi"], weather_data["dhi"], albedo=rho)
POA_pvlib = data["poa_global"]

# Plot both over one week
fig, ax = plt.subplots(figsize=(9, 4))
POA_explicit.loc["2023-06-19":"2023-06-25"].plot(ax=ax, label="Explicit way")
POA_pvlib.loc["2023-06-19":"2023-06-25"].plot(ax=ax, label="pvlib", linestyle="--")
weather_data.loc["2023-06-19":"2023-06-25", "ghi"].plot(ax=ax, label="GHI")
plt.ylabel("Irradiance [W/m²]")
plt.legend() # Print the labels

print(f"Annual POA energy, explicit: {round(POA_explicit.sum() / 1000, 1)} kWh/m²")
print(f"Annual POA energy, pvlib: {round(POA_pvlib.sum() / 1000, 1)} kWh/m²")
print(f"Annual GHI energy: {round(weather_data['ghi'].sum() / 1000, 1)} kWh/m²")

#### Exercice, to go further

The isotropic sky is the simplest model. The Perez model adds the brightening around the sun and at the horizon. Calculate the POA with `model="perez"` in `get_total_irradiance` (it needs two more inputs, see below) and compare the annual energies. Which one is larger, and by how many percent?

In [ ]:
# The Perez model needs the extraterrestrial irradiance and the air mass
dni_extra = pvlib.irradiance.get_extra_radiation(weather_data.index)
airmass = pvlib.atmosphere.get_relative_airmass(solar_position["zenith"])

data_perez = get_total_irradiance(beta, azimuth, solar_position["zenith"], solar_position["azimuth"], weather_data["dni"], weather_data["ghi"], weather_data["dhi"],
                                  albedo=rho, model="perez", dni_extra=dni_extra, airmass=airmass)
POA_perez = data_perez["poa_global"]

E_iso = POA_pvlib.sum() / 1000
E_perez = POA_perez.sum() / 1000
print(f"Isotropic: {round(E_iso, 1)} kWh/m², Perez: {round(E_perez, 1)} kWh/m²")
print(f"Isotropic relative to Perez: {round(100 * (E_iso / E_perez - 1), 2)} %") # Negative: the isotropic sky under-estimates the diffuse on a tilted plane